In [1]:
import mujoco
import numpy as np

model = mujoco.MjModel.from_xml_path("/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/polistamp/polistamp.xml")
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

trajectory_handle = []
trajectory = []

# ID 
site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'target_maniglia')
joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'giunto_porta')
qpos_adr = model.jnt_qposadr[joint_id]
handle_joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'giunto_maniglia')
handle_qpos_adr = model.jnt_qposadr[handle_joint_id]

# Joint Coordinates
joint_center = data.xanchor[joint_id].copy()
Xj, Yj = joint_center[0], joint_center[1]

pos_handle = data.site_xpos[site_id].copy()
Xh, Yh = pos_handle[0], pos_handle[1]

# calculate radius
radius = np.sqrt((Xh - Xj)**2 + (Yh - Yj)**2)
# save points
parameters = np.array([[Xj, Yj, radius]])
np.savetxt(f"/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/polistamp/traiettorie/parametri_cerchio.csv", 
               parameters, delimiter=",", header="Centro_X,Centro_Y,Raggio", comments='')

# Handle coordinates
handle_joint_center = data.xanchor[handle_joint_id].copy()
Xjm, Yjm = handle_joint_center[0], handle_joint_center[1]

pos_target_handle = data.site_xpos[site_id].copy()
Xhm, Yhm = pos_target_handle[0], pos_target_handle[1]

# calculate radius
handle_radius = np.sqrt((Xhm - Xjm)**2 + (Yhm - Yjm)**2)

handle_parameters = np.array([[Xjm, Yjm, handle_radius]])
np.savetxt(f"/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/polistamp/traiettorie/parametri_cerchio_maniglia.csv", 
               handle_parameters, delimiter=",", header="Centro_X,Centro_Y,Raggio", comments='')


# open handle
handle_steps = 100
target_handle_angles = np.linspace(0, np.deg2rad(-10), handle_steps)

mujoco.mj_resetData(model, data)

for angle in target_handle_angles:
    data.qpos[handle_qpos_adr] = angle 
    data.qpos[qpos_adr] = 0.0
    mujoco.mj_forward(model, data) 
    
    pos = data.site_xpos[site_id].copy()

    quat = np.zeros(4)
    mujoco.mju_mat2Quat(quat, data.site_xmat[site_id])
    
    trajectory_handle.append(np.concatenate([pos, quat]))

# open door
steps = 1000
target_angles = np.linspace(0, np.deg2rad(170), steps)

mujoco.mj_resetData(model, data)

for angle in target_angles:
    data.qpos[handle_qpos_adr] = np.deg2rad(-10)
    data.qpos[qpos_adr] = angle 

    mujoco.mj_forward(model, data)
    
    # (x, y, z)
    pos = data.site_xpos[site_id].copy()

    quat = np.zeros(4)
    mujoco.mju_mat2Quat(quat, data.site_xmat[site_id])
    
    trajectory.append(np.concatenate([pos, quat]))

# save
np.save('traiettoria_apertura_maniglia.npy', np.array(trajectory_handle))
np.savetxt("/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/polistamp/traiettorie/traiettoria_apertura_maniglia.csv", trajectory_handle, delimiter=",")
np.save('traiettoria_apertura.npy', np.array(trajectory))
np.savetxt("/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/polistamp/traiettorie/traiettoria_apertura.csv", trajectory, delimiter=",")
print(f"trajectory of {len(trajectory_handle)} points saved.")
print(f"trajectory of {len(trajectory)} points saved.")

trajectory of 100 points saved.
trajectory of 1000 points saved.
